# DistilBERT fine-tuning — ticket triage (queue + priority)

Runs on Kaggle GPU. Reads labeled tickets straight from Neon (the same `redacted_body` the ONNX baseline trains on), fine-tunes two DistilBERT classifiers (queue, priority), and logs one MLflow run with both macro-F1 metrics. Writes `mlflow_run_id.txt` into the kernel output so the Dagster promotion job (`read_kaggle_run_result_op`) can find it.

Requires these Kaggle secrets to be set on this notebook: `NEON_DATABASE_URL`, `MLFLOW_TRACKING_URI`, `DAGSHUB_USER`, `DAGSHUB_TOKEN`.

In [ ]:
!pip install -q mlflow dagshub accelerate evaluate sqlalchemy psycopg2-binary

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
NEON_DATABASE_URL = secrets.get_secret("NEON_DATABASE_URL")
MLFLOW_TRACKING_URI = secrets.get_secret("MLFLOW_TRACKING_URI")
DAGSHUB_USER = secrets.get_secret("DAGSHUB_USER")
DAGSHUB_TOKEN = secrets.get_secret("DAGSHUB_TOKEN")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(NEON_DATABASE_URL)
query = "SELECT redacted_body, true_queue, true_priority FROM tickets WHERE true_queue IS NOT NULL"
df = pd.read_sql(query, engine)
df = df.dropna(subset=["redacted_body", "true_queue", "true_priority"])
print(df.shape)
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

queue_encoder = LabelEncoder()
priority_encoder = LabelEncoder()

df["queue_label"] = queue_encoder.fit_transform(df["true_queue"])
df["priority_label"] = priority_encoder.fit_transform(df["true_priority"])

train_df, eval_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["queue_label"]
)


In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import DistilBertTokenizerFast

TOKENIZER = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")


class TicketDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = TOKENIZER(
            list(texts), truncation=True, padding=True, max_length=256
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1_macro": f1_score(labels, preds, average="macro")}


# `eval_strategy` was renamed from `evaluation_strategy` in newer transformers
# releases -- check the installed version on Kaggle if this errors
def train_classifier(label_column: str, num_labels: int, output_dir: str):
    train_dataset = TicketDataset(train_df["redacted_body"], train_df[label_column])
    eval_dataset = TicketDataset(eval_df["redacted_body"], eval_df[label_column])

    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=num_labels
    )

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    trainer.save_model(output_dir)
    TOKENIZER.save_pretrained(output_dir)

    return metrics["eval_f1_macro"], output_dir


In [ ]:
queue_f1, queue_dir = train_classifier(
    "queue_label", len(queue_encoder.classes_), "/kaggle/working/model/queue"
)
priority_f1, priority_dir = train_classifier(
    "priority_label", len(priority_encoder.classes_), "/kaggle/working/model/priority"
)

print(f"queue_f1_macro={queue_f1:.3f}  priority_f1_macro={priority_f1:.3f}")


In [ ]:
import json
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("ticket-triage-distilbert")

with mlflow.start_run(run_name="distilbert_queue_priority") as run:
    mlflow.log_params(
        {
            "model": "distilbert-base-uncased",
            "epochs": 3,
            "train_rows": len(train_df),
            "eval_rows": len(eval_df),
        }
    )
    mlflow.log_metrics(
        {"queue_f1_macro": queue_f1, "priority_f1_macro": priority_f1}
    )
    run_id = run.info.run_id

# only metrics/params go to MLflow (DagsHub storage is shared with DVC --
# see README limitations); the actual checkpoints travel via Kaggle kernel
# output only, pulled down by dagster_project/kaggle_bridge.py
with open("/kaggle/working/mlflow_run_id.txt", "w") as f:
    f.write(run_id)

label_maps = {
    "queue_classes": queue_encoder.classes_.tolist(),
    "priority_classes": priority_encoder.classes_.tolist(),
}
with open("/kaggle/working/label_maps.json", "w") as f:
    json.dump(label_maps, f)

print(f"logged mlflow run {run_id}")


## Kernel output contract

`/kaggle/working/` after this notebook runs contains:
- `model/queue/`, `model/priority/` — fine-tuned DistilBERT checkpoints + tokenizer
- `mlflow_run_id.txt` — read by `read_kaggle_run_result_op`
- `label_maps.json` — class-index mappings, needed when exporting to ONNX

ONNX export of both checkpoints (via `src.models.onnx_export.export_transformer_to_onnx`, using `optimum`) happens locally after pulling, not here — keeps this notebook GPU-only and ONNX conversion CPU-only, per the project's local-machine constraint.